In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

##Parameters


In [0]:
# #Catalog name
# catalog = "workspace"

# #Key Cols List
# key_cols = "['flight_id']"
# key_cols_list = eval(key_cols)

# #CDC Column
# cdc_col = "modifyDate"

# #Back Dated Refresh
# backdated_refresh = ""

# #Source Object
# source_object = "silver_flights"

# #Source Schema
# source_schema = "silver"

# #TARGET Schema
# target_schema = "gold"

# #Target Object
# target_object = "Dimflights"

# #Surrogate Key 
# surrogate_key = "DimFlightsKey"

In [0]:
#Catalog name
catalog = "workspace"

#Key Cols List
key_cols = "['airport_id']"
key_cols_list = eval(key_cols)

#CDC Column
cdc_col = "modifyDate"

#Back Dated Refresh
backdated_refresh = ""

#Source Object
source_object = "silver_airports"

#Source Schema
source_schema = "silver"

#TARGET Schema
target_schema = "gold"

#Target Object
target_object = "DimAirports"

#Surrogate Key 
surrogate_key = "DimAirportsKey"

In [0]:
# #Catalog name
# catalog = "workspace"

# #Key Cols List
# key_cols = "['passenger_id']"
# key_cols_list = eval(key_cols)

# #CDC Column
# cdc_col = "modifyDate"

# #Back Dated Refresh
# backdated_refresh = ""

# #Source Object
# source_object = "silver_passengers"

# #Source Schema
# source_schema = "silver"

# #TARGET Schema
# target_schema = "gold"

# #Target Object
# target_object = "DimPassengers"

# #Surrogate Key 
# surrogate_key = "DimPassengersKey"

In [0]:
key_cols_list

['airport_id']

## INCREMENTAL DATA INGESTION##

####Last Load Date

In [0]:
#No Back dated Refresh
if len(backdated_refresh) == 0:
    
    #if table exists in the destination
    if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

        last_load = spark.sql(f"SELECT MAX({cdc_col}) FROM {catalog}.{target_schema}.{target_object}").collect()[0][0]
    else:
        last_load = "1900-01-01 00:00:00"  
#Yes Back Dates Refresh
else:
    last_load = backdated_refresh
#test the last load
last_load

'1900-01-01 00:00:00'

In [0]:
df_src = spark.sql(f"SELECT * FROM {source_schema}.{source_object} WHERE {cdc_col} > '{last_load}'")

In [0]:
df_src.display()

airport_id,airport_name,city,country,modifyDate
A001,Gregoryland International Airport,Kristenmouth,South Georgia and the South Sandwich Islands,2025-12-23T00:09:59.580Z
A002,East Kristin International Airport,North Michaelview,Bosnia and Herzegovina,2025-12-23T00:09:59.580Z
A003,Brownland International Airport,Samuelville,Costa Rica,2025-12-23T00:09:59.580Z
A004,Meghanton International Airport,Andrewsmouth,Macedonia,2025-12-23T00:09:59.580Z
A005,East Aaron International Airport,Davishaven,Monaco,2025-12-23T00:09:59.580Z
A006,Michaelburgh International Airport,East Blake,Iceland,2025-12-23T00:09:59.580Z
A007,West Jennifer International Airport,Jillianstad,Libyan Arab Jamahiriya,2025-12-23T00:09:59.580Z
A008,Port Craig International Airport,New Lisa,French Southern Territories,2025-12-23T00:09:59.580Z
A009,New Joshuafurt International Airport,Port Jamiehaven,Pitcairn Islands,2025-12-23T00:09:59.580Z
A010,Thompsontown International Airport,Murraychester,Ireland,2025-12-23T00:09:59.580Z


#### OLD VS NEW RECORDS


In [0]:

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

  #Key Columns String for incremental
  key_cols_string_incremental = ', '.join(key_cols_list)

  df_trg = spark.sql(f"SELECT {key_cols_string_incremental}, {surrogate_key}, create_date, update_date FROM {catalog}.{target_schema}.{target_object} ")
else:
  #Key Columns String for Initial

  
  ##how to unlist and remove the double quotations between the list items(in our case, key_cols_list), we apply a join so a list ["aa", "bb", "cc"] becomes "aa bb cc", and then we insert comma followed by a space right after to enable it to seperate each key column with a comma and a space 
  key_cols_string_init = [f"'' AS {i}" for i in key_cols_list]
  key_cols_string_init = ', '.join(key_cols_string_init)
    
  df_trg = spark.sql(f"""SELECT {key_cols_string_init}, cast('0' AS INT) AS {surrogate_key}, CAST('1900-01-01 00:00:00' AS timestamp) AS create_date, CAST('1900-01-01 00:00:00' AS timestamp) AS update_date WHERE 1=0 """)

  

In [0]:
df_trg.display()

airport_id,DimAirportsKey,create_date,update_date


**JOIN CONDITION**

In [0]:
join_condition = ' AND '.join([f"src.{i} = trg.{i}" for i in key_cols_list])

In [0]:
df_src.createOrReplaceTempView("src")
df_trg.createOrReplaceTempView("trg")


df_join = spark.sql(
    f"""
    SELECT src.*,
           trg.{surrogate_key}, 
           trg.create_date,
           trg.update_date
    FROM src
    LEFT JOIN trg
    ON {join_condition}
    """
)

In [0]:
df_join.display()

airport_id,airport_name,city,country,modifyDate,DimAirportsKey,create_date,update_date
A001,Gregoryland International Airport,Kristenmouth,South Georgia and the South Sandwich Islands,2025-12-23T00:09:59.580Z,null,null,null
A002,East Kristin International Airport,North Michaelview,Bosnia and Herzegovina,2025-12-23T00:09:59.580Z,null,null,null
A003,Brownland International Airport,Samuelville,Costa Rica,2025-12-23T00:09:59.580Z,null,null,null
A004,Meghanton International Airport,Andrewsmouth,Macedonia,2025-12-23T00:09:59.580Z,null,null,null
A005,East Aaron International Airport,Davishaven,Monaco,2025-12-23T00:09:59.580Z,null,null,null
A006,Michaelburgh International Airport,East Blake,Iceland,2025-12-23T00:09:59.580Z,null,null,null
A007,West Jennifer International Airport,Jillianstad,Libyan Arab Jamahiriya,2025-12-23T00:09:59.580Z,null,null,null
A008,Port Craig International Airport,New Lisa,French Southern Territories,2025-12-23T00:09:59.580Z,null,null,null
A009,New Joshuafurt International Airport,Port Jamiehaven,Pitcairn Islands,2025-12-23T00:09:59.580Z,null,null,null
A010,Thompsontown International Airport,Murraychester,Ireland,2025-12-23T00:09:59.580Z,null,null,null


In [0]:
#OLD RECORDS
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())
#NEW RECORDS
df_new = df_join.filter(col(f'{surrogate_key}').isNull())



##ENRICHING DFS

**Preparing DF_OLD**

In [0]:
df_old_enr = df_old.withColumn('update_date',current_timestamp())

In [0]:
df_old_enr.display()

airport_id,airport_name,city,country,modifyDate,DimAirportsKey,create_date,update_date


**Preparing df_new**

In [0]:
df_new.display()

airport_id,airport_name,city,country,modifyDate,DimAirportsKey,create_date,update_date
A001,Gregoryland International Airport,Kristenmouth,South Georgia and the South Sandwich Islands,2025-12-23T00:09:59.580Z,null,null,null
A002,East Kristin International Airport,North Michaelview,Bosnia and Herzegovina,2025-12-23T00:09:59.580Z,null,null,null
A003,Brownland International Airport,Samuelville,Costa Rica,2025-12-23T00:09:59.580Z,null,null,null
A004,Meghanton International Airport,Andrewsmouth,Macedonia,2025-12-23T00:09:59.580Z,null,null,null
A005,East Aaron International Airport,Davishaven,Monaco,2025-12-23T00:09:59.580Z,null,null,null
A006,Michaelburgh International Airport,East Blake,Iceland,2025-12-23T00:09:59.580Z,null,null,null
A007,West Jennifer International Airport,Jillianstad,Libyan Arab Jamahiriya,2025-12-23T00:09:59.580Z,null,null,null
A008,Port Craig International Airport,New Lisa,French Southern Territories,2025-12-23T00:09:59.580Z,null,null,null
A009,New Joshuafurt International Airport,Port Jamiehaven,Pitcairn Islands,2025-12-23T00:09:59.580Z,null,null,null
A010,Thompsontown International Airport,Murraychester,Ireland,2025-12-23T00:09:59.580Z,null,null,null


In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    max_surrogate_key = spark.sql(f"""SELECT max({surrogate_key}) FROM {catalog}.{target_schema}.{target_object}""").collect()[0][0]
    df_new_enr = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key) + lit(1) + monotonically_increasing_id())\
        .withColumn('create_date',current_timestamp())\
        .withColumn('update_date',current_timestamp())
else:
    max_surrogate_key = 0
    df_new_enr = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key) + lit(1) + monotonically_increasing_id())\
        .withColumn('create_date',current_timestamp())\
        .withColumn('update_date',current_timestamp())
        


**UNION OF OLD AND NEW RECORDS**

In [0]:
df_union = df_old_enr.unionByName(df_new_enr)


In [0]:
df_union.display()

airport_id,airport_name,city,country,modifyDate,DimAirportsKey,create_date,update_date
A001,Gregoryland International Airport,Kristenmouth,South Georgia and the South Sandwich Islands,2025-12-23T00:09:59.580Z,1,2026-01-07T23:51:08.310Z,2026-01-07T23:51:08.310Z
A002,East Kristin International Airport,North Michaelview,Bosnia and Herzegovina,2025-12-23T00:09:59.580Z,2,2026-01-07T23:51:08.310Z,2026-01-07T23:51:08.310Z
A003,Brownland International Airport,Samuelville,Costa Rica,2025-12-23T00:09:59.580Z,3,2026-01-07T23:51:08.310Z,2026-01-07T23:51:08.310Z
A004,Meghanton International Airport,Andrewsmouth,Macedonia,2025-12-23T00:09:59.580Z,4,2026-01-07T23:51:08.310Z,2026-01-07T23:51:08.310Z
A005,East Aaron International Airport,Davishaven,Monaco,2025-12-23T00:09:59.580Z,5,2026-01-07T23:51:08.310Z,2026-01-07T23:51:08.310Z
A006,Michaelburgh International Airport,East Blake,Iceland,2025-12-23T00:09:59.580Z,6,2026-01-07T23:51:08.310Z,2026-01-07T23:51:08.310Z
A007,West Jennifer International Airport,Jillianstad,Libyan Arab Jamahiriya,2025-12-23T00:09:59.580Z,7,2026-01-07T23:51:08.310Z,2026-01-07T23:51:08.310Z
A008,Port Craig International Airport,New Lisa,French Southern Territories,2025-12-23T00:09:59.580Z,8,2026-01-07T23:51:08.310Z,2026-01-07T23:51:08.310Z
A009,New Joshuafurt International Airport,Port Jamiehaven,Pitcairn Islands,2025-12-23T00:09:59.580Z,9,2026-01-07T23:51:08.310Z,2026-01-07T23:51:08.310Z
A010,Thompsontown International Airport,Murraychester,Ireland,2025-12-23T00:09:59.580Z,10,2026-01-07T23:51:08.310Z,2026-01-07T23:51:08.310Z


##**UPSERT**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
  dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
  dlt_obj.alias("trg").merge(df_union.alias("src"), f"trg.{surrogate_key} = src.{surrogate_key}")\
    .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
    .whenNotMatchedInsertAll()\
    .execute()


else:
  df_union.write.format("delta").mode("append")\
    .saveAsTable(f"{catalog}.{target_schema}.{target_object}")

